# selfplay_mcts_v4.ipynb -- Stage 5: ResNet (AlphaZero architecture)

Stage 5 replaces the DualNet MLP trunk with a residual network. This is the
actual AlphaZero architecture at small scale (5 residual blocks, 128 filters).
Everything else -- MCTS, dual loss, replay buffer, logging -- is identical to Stage 4.

| Dimension | Stage 4 (selfplay_mcts_v3) | Stage 5 (this notebook) |
|---|---|---|
| Network | DualNet(768, hidden=256) | ResNet(768, n_filters=128, n_blocks=5) |
| Trunk | 2-layer MLP shared | 5 residual blocks x 128 filters |
| Heads | value + policy (same) | value + policy (same) |
| Loss | MSE + CE (same) | MSE + CE (same) |
| Parameters | ~250K | ~660K |

This is the target architecture for the Chess Learning project.
Run this after Stage 4 has converged to get a sense of the incremental benefit
of the residual trunk over the flat MLP.

In [1]:
# Install / imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess", "torch"], check=False)

import os, sqlite3, json, time, random
from collections import deque
from datetime import datetime

import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR

import engine_nn_v2 as eng

PLANE_NAMES = eng.PLANE_NAMES
print(f"Plane names ({len(PLANE_NAMES)}): {PLANE_NAMES}")

Plane names (12): ['P', 'N', 'B', 'R', 'Q', 'K', 'p', 'n', 'b', 'r', 'q', 'k']


In [2]:
# Configuration
RUN_ID             = 33
N_GAMES            = 80
N_SIMULATIONS      = 200          # MCTS simulations per move
REPLAY_BUFFER_SIZE = 10_000
BATCH_SIZE         = 64
LR                 = 1e-3
WEIGHT_DECAY       = 1e-4
LR_MILESTONES      = [200, 500]   # absolute game counts where LR drops 10x
LR_GAMMA           = 0.1
TEMPERATURE        = 1.0          # sample proportional to visit counts during training
CHECKPOINT_EVERY   = 10
RESIGN_THRESHOLD   = 0.9        # resign when value < -this
RESIGN_CONSECUTIVE = 4          # consecutive half-moves below threshold
RESUME             = False

DB_PATH    = "../../data/chess_learning.db"
MODEL_DIR  = "../../models"
MODEL_PATH = f"{MODEL_DIR}/mcts_v4_run_{RUN_ID}.pt"

In [3]:
# Database setup
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# policy_loss column added in Stage 4 -- safe to skip if already present
try:
    cur.execute("ALTER TABLE game_records ADD COLUMN policy_loss REAL")
    con.commit()
    print("Added policy_loss column to game_records")
except Exception:
    pass  # Column already exists

# Run metadata
NOTES          = f"mcts_v4_ResNet_768bitboard_5blocks_128filters n_simulations={N_SIMULATIONS} batch={BATCH_SIZE} replay={REPLAY_BUFFER_SIZE}"
ENGINE_VERSION = "engine_nn_v2"

existing = cur.execute("SELECT run_id FROM runs WHERE run_id = ?", (RUN_ID,)).fetchone()
if existing and not RESUME:
    raise RuntimeError(f"run_id={RUN_ID} already exists. Set RESUME=True or choose a new RUN_ID.")
if not existing:
    cur.execute(
        "INSERT INTO runs (run_id, started_at, depth, learning_rate, move_limit, "
        "notes, last_updated, engine_version) VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        (RUN_ID, datetime.now().isoformat(), 0, LR, None,
         NOTES, datetime.now().isoformat(), ENGINE_VERSION)
    )
    con.commit()
    print(f"Created run_id={RUN_ID}")
else:
    print(f"Resuming run_id={RUN_ID}")

row = cur.execute("SELECT MAX(game_number) FROM games WHERE run_id = ?", (RUN_ID,)).fetchone()[0]
start_game = (row or 0) + 1
print(f"Starting at game_number={start_game}")

Created run_id=33
Starting at game_number=1


In [4]:
# Network, optimizer, replay buffer
os.makedirs(MODEL_DIR, exist_ok=True)

if RESUME and os.path.exists(MODEL_PATH):
    net = eng.load_model(MODEL_PATH, net_class=eng.ResNet,
                         input_dim=768, n_filters=128, n_blocks=5)
    print(f"Loaded model from {MODEL_PATH}")
else:
    net = eng.ResNet(input_dim=768, n_filters=128, n_blocks=5)
    n_params = sum(p.numel() for p in net.parameters())
    print(f"Initialized fresh ResNet(768, n_filters=128, n_blocks=5) -- {n_params:,} parameters")

optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

session_milestones = [max(1, m - start_game + 1) for m in LR_MILESTONES if m >= start_game]
scheduler = MultiStepLR(optimizer, milestones=session_milestones, gamma=LR_GAMMA)
print(f"LR milestones (session-relative): {session_milestones}")

# Replay buffer stores (board_tensor, value_target, legal_moves, policy_target)
replay_buffer = deque(maxlen=REPLAY_BUFFER_SIZE)

Initialized fresh ResNet(768, n_filters=128, n_blocks=5) -- 800,257 parameters
LR milestones (session-relative): [200, 500]


In [5]:
# Self-play game using MCTS with ResNet policy priors
def play_game(net, n_simulations=N_SIMULATIONS, temperature=TEMPERATURE,
              resign_threshold=RESIGN_THRESHOLD, resign_consecutive=RESIGN_CONSECUTIVE):
    # Returns: examples [(board_tensor, value_target, legal_moves, policy_target)],
    #          board, n_halfmoves, terminated_by, resigned_turn
    board           = chess.Board()
    positions       = []
    consecutive_bad = 0
    terminated_by   = "natural"
    resigned_turn   = None

    net.eval()
    while not board.is_game_over(claim_draw=True):
        tensor = eng.board_to_tensor(board)

        # Resignation check (ResNet returns value + policy)
        with torch.no_grad():
            _, val_out = net(tensor.unsqueeze(0))
            lv = val_out.squeeze().item()
        if lv < -resign_threshold:
            consecutive_bad += 1
            if consecutive_bad >= resign_consecutive:
                terminated_by = "resignation"
                resigned_turn = board.turn
                break
        else:
            consecutive_bad = 0

        legal_moves = list(board.legal_moves)
        positions.append((tensor, board.turn, legal_moves, None))

        move, visit_counts = eng.mcts_search(
            board, net,
            n_simulations=n_simulations,
            temperature=temperature,
            # ResNet uses its policy head as priors; board_to_input defaults to board_to_tensor
        )

        total = sum(visit_counts.values()) or 1
        policy_target = torch.tensor(
            [visit_counts.get(m, 0) / total for m in legal_moves],
            dtype=torch.float32
        )
        positions[-1] = (tensor, board.turn, legal_moves, policy_target)
        board.push(move)

    if terminated_by == "resignation":
        white_val = -1.0 if resigned_turn == chess.WHITE else 1.0
    else:
        result    = board.result(claim_draw=True)
        white_val = {"1-0": 1.0, "0-1": -1.0}.get(result, 0.0)

    examples = [
        (tensor,
         white_val if turn == chess.WHITE else -white_val,
         legal_moves,
         policy_target)
        for tensor, turn, legal_moves, policy_target in positions
    ]
    return examples, board, len(positions), terminated_by, resigned_turn

In [6]:
# Dual-loss training step (identical to Stage 4)
def train_step(net, optimizer, batch):
    net.train()
    optimizer.zero_grad()

    total_value_loss  = torch.tensor(0.0)
    total_policy_loss = torch.tensor(0.0)

    for board_tensor, value_target, legal_moves, policy_target in batch:
        policy_probs, value = net.forward(board_tensor, legal_moves)

        v_loss = F.mse_loss(value, torch.tensor(value_target, dtype=torch.float32))
        p_loss = -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum()

        total_value_loss  = total_value_loss  + v_loss
        total_policy_loss = total_policy_loss + p_loss

    n    = len(batch)
    loss = (total_value_loss + total_policy_loss) / n
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item(), (total_value_loss / n).item(), (total_policy_loss / n).item()

def eval_policy_loss(net, examples):
    if not examples:
        return 0.0
    net.eval()
    total = 0.0
    with torch.no_grad():
        for board_tensor, _, legal_moves, policy_target in examples:
            policy_probs, _ = net.forward(board_tensor, legal_moves)
            total += -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum().item()
    return total / len(examples)

In [7]:
# Logging helpers
def board_to_pgn_str(board, game_number):
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "MCTS v4 Self-Play"
    game.headers["Round"] = str(game_number)
    return str(game)

def log_game(game_number, board, n_halfmoves, duration_s, policy_loss,
             terminated_by="natural", resigned_turn=None):
    if terminated_by == "resignation":
        outcome_str = "0-1" if resigned_turn == chess.WHITE else "1-0"
    else:
        result      = board.result(claim_draw=True)
        outcome_str = {"1-0": "1-0", "0-1": "0-1"}.get(result, "1/2-1/2")
    cur.execute(
        "INSERT INTO games (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (RUN_ID, game_number, outcome_str, n_halfmoves, round(duration_s, 2), terminated_by)
    )
    cur.execute(
        "INSERT INTO game_records (run_id, game_number, pgn, policy_loss) VALUES (?, ?, ?, ?)",
        (RUN_ID, game_number, board_to_pgn_str(board, game_number), round(policy_loss, 6))
    )
    con.commit()
    return outcome_str

def log_weights(game_number, prev_weights):
    attr        = eng.attribution_weights(chess.Board(), net)
    scalar_attr = {k: v for k, v in attr.items() if isinstance(v, float)}
    for name, val in scalar_attr.items():
        cur.execute(
            "INSERT INTO weights (run_id, game_number, feature_name, weight_value) "
            "VALUES (?,?,?,?)",
            (RUN_ID, game_number, name, round(float(val), 6))
        )
        if prev_weights and name in prev_weights:
            cur.execute(
                "INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) "
                "VALUES (?,?,?,?)",
                (RUN_ID, game_number, name, round(float(val) - prev_weights[name], 6))
            )
    con.commit()
    return scalar_attr

def log_feature_stats(game_number, outcome_str, examples):
    if not examples:
        return
    stacked     = torch.stack([x[0] for x in examples]).view(-1, 12, 64)
    plane_means = stacked.mean(dim=0).sum(dim=1).tolist()
    cur.executemany(
        "INSERT INTO feature_stats (run_id, game_number, outcome, feature_name, mean_value) "
        "VALUES (?,?,?,?,?)",
        [(RUN_ID, game_number, outcome_str, name, round(plane_means[i], 6))
         for i, name in enumerate(PLANE_NAMES)]
    )
    con.commit()

In [8]:
# Training loop
prev_weights   = None
decisive_count = 0

print(f"MCTS v4  RUN_ID={RUN_ID}  simulations={N_SIMULATIONS}  "
      f"games {start_game}-{start_game + N_GAMES - 1}")
print("-" * 80)

for game_idx in range(N_GAMES):
    game_number = start_game + game_idx
    t0 = time.time()

    examples, board, n_halfmoves, terminated_by, resigned_turn = play_game(net)
    replay_buffer.extend(examples)
    duration = time.time() - t0

    game_policy_loss = eval_policy_loss(net, examples)

    loss, v_loss, p_loss = None, None, None
    if len(replay_buffer) >= BATCH_SIZE:
        batch                = random.sample(replay_buffer, BATCH_SIZE)
        loss, v_loss, p_loss = train_step(net, optimizer, batch)
    scheduler.step()

    outcome_str = log_game(game_number, board, n_halfmoves, duration, game_policy_loss,
                           terminated_by=terminated_by, resigned_turn=resigned_turn)
    log_feature_stats(game_number, outcome_str, examples)
    if outcome_str != "1/2-1/2":
        decisive_count += 1

    if game_number % CHECKPOINT_EVERY == 0:
        prev_weights = log_weights(game_number, prev_weights)
        eng.save_model(net, MODEL_PATH)

    loss_str     = f"{loss:.4f}" if loss is not None else "  n/a "
    p_loss_str   = f"{p_loss:.4f}" if p_loss is not None else "  n/a "
    decisive_pct = decisive_count / (game_idx + 1) * 100
    print(f"G{game_number:4d} | {outcome_str:7s} | {n_halfmoves:3d} ply | "
          f"{duration:5.1f}s | loss={loss_str} p={p_loss_str} | "
          f"buf={len(replay_buffer):5d} | decisive={decisive_pct:4.1f}%")

print("-" * 80)
print(f"Done. Model saved to {MODEL_PATH}")
con.close()

MCTS v4  RUN_ID=33  simulations=200  games 1-80
--------------------------------------------------------------------------------
G   1 | 1/2-1/2 | 174 ply |  68.0s | loss=2.8128 p=2.8103 | buf=  174 | decisive= 0.0%
G   2 | 1/2-1/2 | 230 ply |  91.9s | loss=3.0717 p=3.0705 | buf=  404 | decisive= 0.0%
G   3 | 1/2-1/2 | 498 ply | 250.1s | loss=2.9827 p=2.9819 | buf=  902 | decisive= 0.0%
G   4 | 1/2-1/2 | 298 ply | 126.5s | loss=2.8979 p=2.8977 | buf= 1200 | decisive= 0.0%
G   5 | 1/2-1/2 | 285 ply | 124.5s | loss=3.0110 p=3.0109 | buf= 1485 | decisive= 0.0%
G   6 | 1/2-1/2 | 310 ply | 134.8s | loss=2.7008 p=2.7006 | buf= 1795 | decisive= 0.0%
G   7 | 1/2-1/2 | 353 ply | 165.5s | loss=2.8052 p=2.8050 | buf= 2148 | decisive= 0.0%
G   8 | 1/2-1/2 | 254 ply | 105.0s | loss=2.9016 p=2.9015 | buf= 2402 | decisive= 0.0%
G   9 | 1/2-1/2 | 332 ply | 157.5s | loss=2.8971 p=2.8971 | buf= 2734 | decisive= 0.0%
G  10 | 1/2-1/2 | 377 ply | 183.6s | loss=2.9791 p=2.9791 | buf= 3111 | decisive= 0.0%
G